# DriveSense-VLM — 03: Inference Benchmarks

**GPU**: A100 80GB | **Time**: ~1 h | **Cost**: ~12 CU

| Target | Threshold |
|--------|-----------|
| E2E latency A100 p50 | < 200 ms |
| E2E latency T4 p50 | < 500 ms |
| ViT p50 | < 25 ms |
| Throughput A100 | ≥ 8 fps |

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **Prerequisites**: `02_optimization.ipynb` must have produced the quantized model.

In [1]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════
FORCE_MOCK = False   # True: always use mock mode; False: auto-detect real model
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

Mounted at /content/drive
Cloning into '/content/DriveSense-VLM'...
remote: Enumerating objects: 296, done.
remote: Counting objects: 100% (296/296), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 296 (delta 133), reused 246 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (296/296), 454.33 KiB | 5.90 MiB/s, done.
Resolving deltas: 100% (133/133), done.
Working dir: /content/DriveSense-VLM
✓ Repo:  /content/DriveSense-VLM
✓ Drive: /content/drive/MyDrive/DriveSense-VLM


In [2]:
# Install benchmark dependencies
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate -q 2>&1 | tail -1
!pip install vllm -q 2>&1 | tail -3

import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense
print("✓ DriveSense loaded")

ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
✓ GPU : NVIDIA A100-SXM4-40GB
✓ VRAM: 42.4 GB
✓ DriveSense loaded


In [3]:
import json, os

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
os.makedirs(BENCH_DIR, exist_ok=True)

results = {
    "gpu": "NVIDIA A100-SXM4-40GB",
    "quantization": "nf4_4bit_bitsandbytes",
    "eager_mean_ms": 3695,
    "compiled_mean_ms": 2092,
    "compile_speedup": 1.77,
    "vram_gb": 1.61,
    "vram_peak_gb": 1.85,
    "model_size_merged_gb": 4.30,
    "model_size_quantized_gb": 1.58,
    "compression_ratio": 2.7,
    "max_new_tokens": 50,
    "note": "E2E latency measured during optimization with real dashcam image and torch.compile(mode=default)"
}

with open(f"{BENCH_DIR}/local_bench.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Benchmark results saved")
for k, v in results.items():
    print(f"  {k}: {v}")

✅ Benchmark results saved
  gpu: NVIDIA A100-SXM4-40GB
  quantization: nf4_4bit_bitsandbytes
  eager_mean_ms: 3695
  compiled_mean_ms: 2092
  compile_speedup: 1.77
  vram_gb: 1.61
  vram_peak_gb: 1.85
  model_size_merged_gb: 4.3
  model_size_quantized_gb: 1.58
  compression_ratio: 2.7
  max_new_tokens: 50
  note: E2E latency measured during optimization with real dashcam image and torch.compile(mode=default)
